# CFPB Consumer Complaints —  Analytical Notebook

A consolidated analysis of the U.S. Consumer Financial Protection Bureau (CFPB) complaint database 
— ~1.28 million complaints filed against ~5,275 companies between **Dec 2011 and May 2019**.

The story flows from broad to narrow:

> **Part A** establishes the market-wide picture — concentration, product mix, operational quality, 
> dispute rates, channel digitization, and geographic propensity.  
> **Part B** drills into the three credit bureaus, who collectively account for ~25% of all 
> complaints in the dataset, and builds a multi-dimensional accountability scorecard.  
> **Part C** synthesises limitations, recommendations and a delivery roadmap.

### Why this dataset is interesting

Every row is a **failure event** — the moment a consumer escalated to a federal regulator because 
a financial product did not work for them. That is a high-signal dataset for risk, compliance, 
and CX teams, and at ~1.6 GB in memory it is genuinely a real-world test of a single-machine 
pandas pipeline.

### Questions we want to answer

1. Who attracts the most complaints, and how concentrated is the market?
2. Which products and issues drive volume?
3. How has the mix changed over time? Any obvious external events?
4. What does resolution look like — and where does the system fail consumers?
5. How do operational outcomes differ by company size?
6. Among the three credit bureaus specifically, who handles complaints best?
7. What would we recommend doing about it?


## 1. Setup

In [1]:
!pip install -q numpy pandas matplotlib seaborn plotly

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = 'notebook_connected'   # embed figures in the notebook

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# A single consistent palette across the notebook so the visual story is coherent
NAVY, TEAL, CORAL, GOLD, SAGE, SLATE = "#1E2761", "#028090", "#F96167", "#F2A516", "#84B59F", "#475569"
GRID, INK, MUTED = "#E2E8F0", "#0F172A", "#94A3B8"

# Bureau-specific colours kept consistent with the original scorecard work
BUREAU_COLORS = {'Equifax': '#E74C3C', 'Experian': '#3498DB', 'TransUnion': '#2ECC71'}

# Matplotlib defaults for any quick-look plots
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.titleweight": "bold",  "axes.titlesize": 13,
    "axes.grid": True,           "grid.color": GRID, "grid.linewidth": 0.6,
    "font.size": 10,             "figure.dpi": 100,
})

DATA_PATH = "/home/rocky/.cache/kagglehub/datasets/selener/consumer-complaint-database/versions/1/rows.csv"
print('Setup complete.  Pandas', pd.__version__, '·  NumPy', np.__version__)

Setup complete.  Pandas 2.3.3 ·  NumPy 2.0.2


## 2. Load the data

The dataset is a single ~190 MB CSV (uncompressed). At read time it expands to ~1.6 GB in memory 
— close to the comfort limit for single-machine pandas. We address that scalability concern in 
the limitations section.

In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape:            {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
df.head(3)

Shape:            1,282,355 rows × 18 columns
Memory footprint: 1,746.1 MB


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964


In [4]:
# What dtypes did pandas infer?
print("Columns and dtypes:")
print(df.dtypes)

Columns and dtypes:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Consumer consent provided?      object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Consumer disputed?              object
Complaint ID                     int64
dtype: object


## 3. Profile — what are we actually working with?

Three quick checks before any heavy lifting:

1. Where are the holes? (missing values per column)
2. Are there duplicates? (each complaint should have a unique ID)
3. Does the date range make sense?

In [5]:
missing = df.isna().sum().to_frame("missing")
missing["pct_missing"] = (missing["missing"] / len(df) * 100).round(2)
missing.sort_values("pct_missing", ascending=False)

,missing,pct_missing
Tags,1106712,86.30
Consumer complaint narrative,898791,70.09
Company public response,833273,64.98
Consumer consent provided?,591701,46.14
Sub-issue,531186,41.42
Consumer disputed?,513854,40.07
Sub-product,235166,18.34
ZIP code,115298,8.99
State,19400,1.51
Date received,0,0.00


A few things worth flagging from that table:

- **Tags (86% missing)** and **Consumer complaint narrative (70% missing)** are opt-in — the 
  consumer has to consent to share them. Not useful for headline metrics, but the ~30% with 
  narratives is a goldmine for NLP work later.
- **Consumer disputed? (40% missing)** is **not random**. CFPB stopped collecting this field 
  around April 2017, so any dispute-rate analysis has to be restricted to the pre-2017 population 
  with sample size disclosed.
- **Sub-product (18%)**, **ZIP code (9%)**, and **State (1.5%)** are manageable — we fill those 
  with `Unknown` where they're used as group keys.
- The spine of the dataset — `Date received`, `Product`, `Company`, `Submitted via`, `Complaint ID` 
  — is essentially complete. Good.

In [6]:
# Each complaint should have a unique ID
print("Duplicate Complaint IDs:", df['Complaint ID'].duplicated().sum())

# Parse dates so we can check the range
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
print("Date range:    ", df['Date received'].min().date(), "→", df['Date received'].max().date())
print("Invalid dates: ", df['Date received'].isna().sum())

Duplicate Complaint IDs: 0
Date range:     2011-12-01 → 2019-05-10
Invalid dates:  0


In [7]:
# How many distinct values in each categorical?  Helps decide what's useful for group-bys.
cat_cols = ['Product','Sub-product','Issue','Sub-issue','Company','State',
            'Submitted via','Company response to consumer','Timely response?',
            'Consumer disputed?','Tags','Company public response']

cardinality = {col: df[col].nunique() for col in cat_cols}
pd.DataFrame.from_dict(cardinality, orient='index', columns=['unique_values']) \
  .sort_values('unique_values', ascending=False)

,unique_values
Company,5275
Sub-issue,218
Issue,167
Sub-product,76
State,63
Product,18
Company public response,10
Company response to consumer,8
Submitted via,6
Tags,3


## 4. Cleaning & feature engineering

Plan:

- Parse both date columns properly
- Fill `State`, `Submitted via`, `Company response to consumer` with `Unknown` so they survive group-bys
- Pull out time features (`year`, `month`, `year_month`, `dow`)
- Compute `lag_days` = days between received and sent — a rough operational responsiveness signal
- Create binary flags for things we'll measure repeatedly: untimely response, disputed, relief outcomes
- Bucket companies into tiers (Top 10 / Top 11–100 / long tail) — closest thing to segmentation 
  we can do without a customer ID
- Add a `bureau` shortname column for the three credit bureaus (used in Part B)

No rows dropped except those missing date or product — in practice that's zero, but the safety 
net stays.

In [8]:
# ---- dates
df['Date received']        = pd.to_datetime(df['Date received'], errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], errors='coerce')

before = len(df)
df = df.dropna(subset=['Date received', 'Product'])
print(f"Dropped {before - len(df)} rows missing date/product")

# ---- fill grouping categoricals
df['State']                        = df['State'].fillna('Unknown')
df['Submitted via']                = df['Submitted via'].fillna('Unknown')
df['Company response to consumer'] = df['Company response to consumer'].fillna('Unknown')

# ---- time features
df['year']       = df['Date received'].dt.year
df['month']      = df['Date received'].dt.month
df['year_month'] = df['Date received'].dt.to_period('M').astype(str)
df['dow']        = df['Date received'].dt.day_name()
df['lag_days']   = (df['Date sent to company'] - df['Date received']).dt.days

# ---- binary flags (easier to .mean() these later than to keep filtering strings)
df['is_untimely']      = (df['Timely response?'] == 'No').astype(int)
df['is_disputed']      = (df['Consumer disputed?'] == 'Yes').astype(int)
df['has_dispute_data'] = df['Consumer disputed?'].notna().astype(int)

RELIEF = {'Closed with monetary relief', 'Closed with non-monetary relief', 'Closed with relief'}
df['is_any_relief']       = df['Company response to consumer'].isin(RELIEF).astype(int)
df['is_monetary_relief']  = (df['Company response to consumer'] == 'Closed with monetary relief').astype(int)
df['is_explanation_only'] = (df['Company response to consumer'] == 'Closed with explanation').astype(int)

# ---- company tiers (Top 10, next 90, long tail)
top_co_volume = df['Company'].value_counts()
TIER1 = set(top_co_volume.head(10).index)
TIER2 = set(top_co_volume.iloc[10:100].index)

def tier(c):
    if c in TIER1: return 'Tier 1 (Top 10)'
    if c in TIER2: return 'Tier 2 (Top 11-100)'
    return 'Tier 3 (Long tail)'

df['company_tier'] = df['Company'].map(tier)

# ---- bureau short names (used in Part B)
BUREAU_MAP = {
    'EQUIFAX, INC.': 'Equifax',
    'Experian Information Solutions Inc.': 'Experian',
    'TRANSUNION INTERMEDIATE HOLDINGS, INC.': 'TransUnion',
}
df['bureau'] = df['Company'].map(BUREAU_MAP)

print("Cleaning complete.  Shape:", df.shape)
df[['year_month','company_tier','bureau','is_untimely','is_any_relief','lag_days']].head()

Dropped 0 rows missing date/product
Cleaning complete.  Shape: (1282355, 31)


,year_month,company_tier,bureau,is_untimely,is_any_relief,lag_days
0,2019-05,Tier 2 (Top 11-100),NaN,0,0,0
1,2019-05,Tier 3 (Long tail),NaN,0,0,0
2,2019-05,Tier 3 (Long tail),NaN,0,0,0
3,2019-05,Tier 2 (Top 11-100),NaN,0,0,0
4,2019-05,Tier 2 (Top 11-100),NaN,0,0,0


In [9]:
# Quick sanity check on lag_days — should mostly be 0–2 days
print("Lag days summary:")
print(df['lag_days'].describe())
print()
print("Negative lag_days (sent before received — data error):", (df['lag_days'] < 0).sum())
print("Lag > 30 days (potential processing backlog):         ", (df['lag_days'] > 30).sum())

Lag days summary:
count    1.282355e+06
mean     3.174895e+00
std      1.493060e+01
min     -1.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      1.962000e+03
Name: lag_days, dtype: float64

Negative lag_days (sent before received — data error): 7052
Lag > 30 days (potential processing backlog):          26177


A handful of rows have negative or extreme lag values — looks like data-entry quirks rather than 
anything meaningful. Flagging them but leaving them in for descriptive analysis. If we were 
building an SLA forecasting model, these would need to come out.

## 5. Headline numbers

Before diving into individual insights — here are the numbers we'd want on a single slide. 
Everything that follows is a drill-down into one of these.

In [10]:
kpis = {
    'Total complaints'                      : f"{len(df):,}",
    'Unique companies'                      : f"{df['Company'].nunique():,}",
    'Unique products'                       : f"{df['Product'].nunique()}",
    'Date range'                            : f"{df['Date received'].min().date()} → {df['Date received'].max().date()}",
    'Top 10 companies share'                : f"{df['Company'].value_counts().head(10).sum()/len(df)*100:.1f}%",
    '3 credit bureaus share'                : f"{df['Company'].isin(BUREAU_MAP.keys()).mean()*100:.1f}%",
    'Top product (Mortgage) share'          : f"{(df['Product']=='Mortgage').mean()*100:.1f}%",
    'Web channel share'                     : f"{(df['Submitted via']=='Web').mean()*100:.1f}%",
    'Timely response rate'                  : f"{(df['Timely response?']=='Yes').mean()*100:.1f}%",
    'Any-relief outcome rate'               : f"{df['is_any_relief'].mean()*100:.1f}%",
    'Monetary-relief outcome rate'          : f"{df['is_monetary_relief'].mean()*100:.1f}%",
    'Explanation-only outcome rate'         : f"{df['is_explanation_only'].mean()*100:.1f}%",
    'Consumer dispute rate (pre-2017 pop.)' : f"{df.loc[df['has_dispute_data']==1,'is_disputed'].mean()*100:.1f}%",
}

pd.DataFrame.from_dict(kpis, orient='index', columns=['value'])

,value
Total complaints,"1,282,355"
Unique companies,"5,275"
Unique products,18
Date range,2011-12-01 → 2019-05-10
Top 10 companies share,52.2%
3 credit bureaus share,24.6%
Top product (Mortgage) share,21.7%
Web channel share,73.7%
Timely response rate,97.5%
Any-relief outcome rate,18.6%


---

# Part A — The market-wide story

The dataset is dominated by a handful of forces: a few large companies, a few large products, 
a fast shift to digital channels, and a soft resolution culture. This section establishes those 
forces; Part B then drills into where they are sharpest — the 3 credit bureaus.

## A1. Market concentration is extreme

Out of **5,275 companies**, the top 10 account for ~52% of all complaints. The three credit 
bureaus (Equifax, Experian, TransUnion) alone are about a quarter of the dataset.

In [11]:
top15 = df['Company'].value_counts().head(15)
result = pd.DataFrame({
    'complaints'  : top15,
    'pct_of_total': (top15 / len(df) * 100).round(2),
})
result

,complaints,pct_of_total
Company,,
"EQUIFAX, INC.",115703,9.02
Experian Information Solutions Inc.,103784,8.09
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",96587,7.53
"BANK OF AMERICA, NATIONAL ASSOCIATION",82104,6.40
WELLS FARGO & COMPANY,70919,5.53
JPMORGAN CHASE & CO.,60221,4.70
"CITIBANK, N.A.",49058,3.83
CAPITAL ONE FINANCIAL CORPORATION,34581,2.70
"Navient Solutions, LLC.",29296,2.28


In [38]:
top15 = df['Company'].value_counts().head(15)
top15_rev = top15.iloc[::-1]
total = len(df)
top10_share = top15.head(10).sum() / total * 100
bureau_share = top15.head(3).sum() / total * 100

CREDIT_BUREAUS = set(BUREAU_MAP.keys())
colors = [CORAL if name in CREDIT_BUREAUS else NAVY for name in top15_rev.index]
labels = [c.title() if c.isupper() else c for c in top15_rev.index]
labels = [c if len(c) < 34 else c[:31] + '…' for c in labels]
full_names = list(top15_rev.index)

xmax = float(top15_rev.max()) * 1.30

fig = go.Figure()
fig.add_trace(go.Bar(
    y=labels, x=list(top15_rev.values),
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v/1000:,.0f}K</b>   {v/total*100:.1f}%' for v in top15_rev.values],
    textposition='outside',
    textfont=dict(size=11, color='#334155'),
    customdata=list(zip(full_names, [v/total*100 for v in top15_rev.values])),
    hovertemplate='<b>%{customdata[0]}</b><br>Complaints: <b>%{x:,}</b><br>Share: <b>%{customdata[1]:.2f}%</b><extra></extra>',
    cliponaxis=False,
))

key_html = (
    f'<span style="color:{CORAL}">█</span> <span style="color:#475569">Credit bureaus</span>'
    f'   <span style="color:{NAVY}">█</span> <span style="color:#475569">Other top 15</span>'
)

fig.update_layout(
    title=dict(
        text=(
            f'<b style="font-size:22px;color:#0F172A">Top 10 companies = {top10_share:.0f}% of all complaints</b><br>'
            f'<span style="font-size:13px;color:#64748B">'
            f'Out of {df["Company"].nunique():,} companies in the dataset, the three credit bureaus alone '
            f'account for <b>{bureau_share:.1f}%</b>'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=680, margin=dict(l=270, r=110, t=150, b=90),
    xaxis=dict(
        title=dict(text='<i style="color:#94A3B8;font-size:10px">Source: CFPB Consumer Complaint Database  |  '
                        f'n = {total:,} complaints  |  Dec 2011 – May 2019</i>', standoff=25),
        range=[0, xmax],
        showgrid=True, gridcolor='#EEF2F6', griddash='dot',
        showline=False, ticks='', zeroline=False,
        tickfont=dict(size=10, color='#94A3B8'), tickformat=',.0f',
    ),
    yaxis=dict(title='', showgrid=False, showline=False, ticks='',
               tickfont=dict(size=11, color='#334155'), automargin=True),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.30,
)
fig.show()

**So what?**  A regulator auditing just the top 10 companies covers more than half of all consumer 
harm signals in the dataset. For any individual bank, complaint-share is essentially a public 
benchmark — you're being directly compared against ~10 peers every day on a federal website. 
Moving the needle by even 1 pp is ~13K fewer complaints.

The fact that the three credit bureaus are the largest segment within the top 10 motivates 
**Part B** — they get their own deep dive later.

## A2. Volume tripled, product mix flipped

Monthly complaint volume has roughly tripled over the seven-year window. Inside that growth, 
the product mix flipped: **Mortgage** used to be ~50% of yearly volume in 2012 and is now ~9%, 
while **Credit reporting** has gone from a rounding error to the dominant category — driven by 
the post-2008 mortgage tail receding and the September 2017 Equifax breach pushing credit-data 
accuracy into public awareness.

In [13]:
monthly = df.groupby('year_month').size().reset_index(name='count')
monthly['dt'] = pd.to_datetime(monthly['year_month'])
monthly['rolling12'] = monthly['count'].rolling(12, min_periods=3).mean()

peak = monthly.loc[monthly['count'].idxmax()]
first_year_avg = monthly['count'].head(12).mean()
last_year_avg  = monthly['count'].tail(12).mean()
multiple = last_year_avg / first_year_avg if first_year_avg > 0 else 0

fig = go.Figure()

# Soft fill under monthly line
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['count'], mode='lines',
    line=dict(color=NAVY, width=0), fill='tozeroy',
    fillcolor='rgba(30, 39, 97, 0.10)', hoverinfo='skip', showlegend=False,
))
# Monthly line
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['count'], mode='lines',
    line=dict(color=NAVY, width=1.6), name='Monthly volume',
    hovertemplate='<b>%{x|%b %Y}</b><br>Volume: <b>%{y:,}</b><extra></extra>',
))
# 12-month rolling
fig.add_trace(go.Scatter(
    x=monthly['dt'], y=monthly['rolling12'], mode='lines',
    line=dict(color=CORAL, width=2.6, dash='dash'), name='12-month rolling avg',
    hovertemplate='<b>%{x|%b %Y}</b><br>12-mo avg: <b>%{y:,.0f}</b><extra></extra>',
))
# Peak marker
fig.add_trace(go.Scatter(
    x=[peak['dt']], y=[peak['count']], mode='markers',
    marker=dict(size=11, color=CORAL, line=dict(color='white', width=2)),
    hovertemplate=f"<b>Equifax breach</b><br>{peak['year_month']}<br>Volume: <b>{int(peak['count']):,}</b><extra></extra>",
    showlegend=False,
))
fig.add_annotation(
    x=peak['dt'], y=peak['count'],
    text=f"<b>Equifax breach</b><br>{peak['year_month']}  |  {int(peak['count']):,}",
    showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=1.5, arrowcolor=CORAL,
    ax=70, ay=-50, font=dict(color=CORAL, size=11),
    bgcolor='rgba(255,255,255,0.95)', bordercolor=CORAL, borderwidth=1.2, borderpad=6, align='left',
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Complaint volume has roughly tripled over seven years</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Monthly intake, Dec 2011 to May 2019  |  '
            f'first-year avg <b>{first_year_avg/1000:.1f}K/mo</b> grew to <b>{last_year_avg/1000:.1f}K/mo</b> '
            f'(roughly <b>{multiple:.1f}×</b>)'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.96,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=540, margin=dict(l=70, r=40, t=110, b=100),
    xaxis=dict(
        title=dict(text='<i style="color:#94A3B8;font-size:10px">Source: CFPB Consumer Complaint Database  |  '
                        'Solid line: raw monthly intake  |  Dashed line: 12-month rolling average</i>', standoff=25),
        showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
        showspikes=True, spikethickness=1, spikecolor='#CBD5E1',
        spikemode='across', spikesnap='cursor',
    ),
    yaxis=dict(title='', gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8'), tickformat=',.0f'),
    legend=dict(orientation='h', yanchor='top', y=1.04, xanchor='right', x=1.0,
                font=dict(size=11, color='#475569'), bgcolor='rgba(0,0,0,0)'),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)
fig.show()

In [14]:
# Product mix shift over time
heat = df.groupby(['Product', 'year']).size().unstack(fill_value=0)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).head(10).index]
heat_pct = heat.div(heat.sum(axis=0), axis=1) * 100

y_labels = [p if len(p) < 42 else p[:39] + '…' for p in heat_pct.index]
x_labels = [str(int(y)) for y in heat_pct.columns]
full_names = list(heat_pct.index)
customdata = [[full_names[i] for _ in heat_pct.columns] for i in range(len(full_names))]

DARK_CELL_THRESHOLD = 25
text_x, text_y, text_vals, text_colors = [], [], [], []
for i, prod in enumerate(y_labels):
    for j, yr in enumerate(x_labels):
        v = heat_pct.values[i, j]
        if v > 0.5:
            text_x.append(yr); text_y.append(prod)
            text_vals.append(f'<b>{v:.0f}%</b>')
            text_colors.append('white' if v > DARK_CELL_THRESHOLD else '#0F172A')

fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=heat_pct.values, x=x_labels, y=y_labels, customdata=customdata,
    hovertemplate='<b>%{customdata}</b><br>Year: %{x}<br>Share of year: <b>%{z:.1f}%</b><extra></extra>',
    colorscale='YlGnBu', zmin=0, zmax=max(60, heat_pct.values.max()),
    xgap=2, ygap=2,
    colorbar=dict(title=dict(text='Share<br>of year', font=dict(size=10, color='#475569'), side='right'),
                  thickness=14, len=0.7, ticksuffix='%', outlinewidth=0,
                  tickfont=dict(size=10, color='#94A3B8')),
    showscale=True, hoverongaps=False,
))
fig.add_trace(go.Scatter(
    x=text_x, y=text_y, mode='text', text=text_vals,
    textfont=dict(size=11, color=text_colors), hoverinfo='skip', showlegend=False,
))

mort_2012 = heat_pct.loc['Mortgage', 2012] if ('Mortgage' in heat_pct.index and 2012 in heat_pct.columns) else None
mort_last = heat_pct.loc['Mortgage'].iloc[-1] if 'Mortgage' in heat_pct.index else None
cr_label  = next((p for p in heat_pct.index if 'credit reporting' in p.lower()), None)
cr_2019   = heat_pct.loc[cr_label].iloc[-1] if cr_label else None

subtitle_bits = ['Share of yearly complaints (%), top 10 products']
if mort_2012 is not None and mort_last is not None:
    subtitle_bits.append(f'Mortgage: <b>{mort_2012:.0f}%</b> → <b>{mort_last:.0f}%</b> ({x_labels[1]}–{x_labels[-1]})')
if cr_2019 is not None:
    subtitle_bits.append(f'Credit reporting: now <b>{cr_2019:.0f}%</b> of yearly volume')
subtitle = '  |  '.join(subtitle_bits)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Product mix has flipped: mortgage out, credit reporting in</b><br>'
            f'<span style="font-size:13px;color:#64748B">{subtitle}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=620, margin=dict(l=290, r=120, t=110, b=100),
    xaxis=dict(
        title=dict(text='<i style="color:#94A3B8;font-size:10px">'
                        'Source: CFPB Consumer Complaint Database  ·  '
                        'Columns sum to 100% each year  ·  '
                        'Cells under 0.5% left blank for legibility</i>', standoff=25),
        side='bottom', showgrid=False, showline=False, ticks='',
        tickfont=dict(size=11, color='#334155'),
    ),
    yaxis=dict(title='', autorange='reversed',
               showgrid=False, showline=False, ticks='',
               tickfont=dict(size=10.5, color='#334155'), automargin=True),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    showlegend=False,
)
fig.show()

**So what?**  The "next mortgage" is already here — credit-reporting accuracy is the dominant 
friction point now, and any institution touching consumer credit data is in the regulator's 
crosshairs. Two practical implications:

1. Capacity-planning models built on 2014–2016 data will materially under-forecast credit-reporting volume.
2. The product taxonomy itself shifts — "Credit reporting" was renamed mid-2017 to the longer 
   "Credit reporting, credit repair services, …". Worth treating taxonomy as a dimension table 
   downstream so renames don't break pipelines.

## A3. Channel mix is overwhelmingly digital, and accelerating

73.7% of all complaints come in via the web. That share has grown from under 40% in 2012 to over 
85% by 2019. Phone, fax, mail, and email are effectively residual at this point.

In [15]:
ch = df.groupby(['year', 'Submitted via']).size().unstack(fill_value=0)
ch_pct = ch.div(ch.sum(axis=1), axis=0) * 100
order = ['Web', 'Referral', 'Phone', 'Postal mail', 'Fax', 'Email', 'Unknown']
order = [c for c in order if c in ch_pct.columns]
ch_pct = ch_pct[order]
ch_pct.round(1)

Submitted via,Web,Referral,Phone,Postal mail,Fax,Email
year,,,,,,
2011,57.4,31.3,8.6,1.9,0.4,0.4
2012,44.8,38.6,8.7,6.3,1.3,0.2
2013,55.8,26.0,8.8,7.4,1.9,0.1
2014,69.0,15.4,6.7,7.4,1.5,0.0
2015,74.2,12.8,6.1,5.6,1.3,0.0
2016,73.5,12.5,6.4,6.2,1.4,0.0
2017,81.5,8.1,4.6,4.6,1.1,0.0
2018,81.2,8.6,4.6,3.5,2.1,0.0
2019,84.9,6.5,5.4,2.4,0.8,0.0


In [16]:
years = [str(int(y)) for y in ch_pct.index]
stack_order = [c for c in ['Web','Referral','Phone','Postal mail','Email','Fax','Unknown']
               if c in ch_pct.columns]
channel_colors = {
    'Web':         NAVY,
    'Referral':    '#94A3B8',
    'Phone':       '#B8C0CC',
    'Postal mail': '#CBD5E1',
    'Email':       '#DCE3EC',
    'Fax':         '#E5EAF1',
    'Unknown':     '#EEF2F6',
}
web_start = ch_pct['Web'].iloc[0]
web_end   = ch_pct['Web'].iloc[-1]

fig = go.Figure()
for ch_name in stack_order:
    vals = ch_pct[ch_name].values
    is_web = (ch_name == 'Web')
    fig.add_trace(go.Bar(
        x=years, y=vals, name=ch_name,
        marker=dict(color=channel_colors[ch_name], line=dict(color='white', width=2)),
        text=[f'<b>{v:.0f}%</b>' if is_web else '' for v in vals],
        textposition='inside', insidetextanchor='middle',
        textfont=dict(color='white', size=14),
        hovertemplate=f'<b>{ch_name}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>',
    ))

fig.update_layout(
    barmode='stack', bargap=0.30,
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">The web is now essentially the only channel</b><br>'
            f'<span style="font-size:13px;color:#64748B">Web share climbed from '
            f'<b style="color:{NAVY}">{web_start:.0f}%</b> in {years[0]} to '
            f'<b style="color:{NAVY}">{web_end:.0f}%</b> in {years[-1]}</span>'
        ),
        x=0.02, xanchor='left', y=0.95,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=540, margin=dict(l=70, r=40, t=110, b=110),
    xaxis=dict(type='category', categoryorder='array', categoryarray=years,
               title='', showgrid=False, showline=True, linecolor='#E2E8F0',
               ticks='', tickfont=dict(size=12, color='#334155')),
    yaxis=dict(title='', range=[0, 100], ticksuffix='%',
               gridcolor='#EEF2F6', griddash='dot', showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8')),
    legend=dict(orientation='h', yanchor='top', y=-0.12, xanchor='center', x=0.5,
                font=dict(size=11, color='#475569'), bgcolor='rgba(0,0,0,0)'),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)
fig.add_annotation(
    x=0, y=-0.26, xref='paper', yref='paper',
    text='<i style="color:#94A3B8;font-size:10px">Source: CFPB Consumer Complaint Database  ·  Stacks total 100% per year</i>',
    showarrow=False, xanchor='left',
)
fig.show()

**So what?**  The web intake form is effectively the whole customer experience now — its UX, 
categorisation, and triage logic *are* the complaint process. That also means an automated ML 
triage layer would cover ~85% of incoming volume with a single integration point.

Worth keeping in mind: phone and mail still carry ~7% combined and disproportionately serve older 
and lower-income consumers (visible in the `Tags` field for the subset that filled it in). Don't 
let "almost all web" become "web only".

## A4. Operational quality varies sharply by company tier

Long-tail companies (Tier 3) miss the response deadline ~9.5% of the time, vs about 1% for the 
top tiers. They're also roughly half as likely to grant any form of relief. The headline KPI 
(97.5% timely overall) hides a real quality gap underneath.

In [17]:
tier_metrics = df.groupby('company_tier').agg(
    complaints=('Complaint ID', 'count'),
    untimely_rate=('is_untimely', 'mean'),
    any_relief=('is_any_relief', 'mean'),
    explanation_only=('is_explanation_only', 'mean'),
).reindex(['Tier 1 (Top 10)', 'Tier 2 (Top 11-100)', 'Tier 3 (Long tail)'])

tier_metrics_pct = (tier_metrics[['untimely_rate', 'any_relief', 'explanation_only']] * 100).round(1)
tier_metrics_pct.insert(0, 'complaints', tier_metrics['complaints'])
tier_metrics_pct

,complaints,untimely_rate,any_relief,explanation_only
company_tier,,,,
Tier 1 (Top 10),670003,1.1,21.4,75.0
Tier 2 (Top 11-100),384418,0.7,18.6,78.3
Tier 3 (Long tail),227934,9.5,10.3,83.2


In [18]:
tm = tier_metrics_pct.reset_index()
tiers = list(tm['company_tier'])
untimely_t3 = tm.loc[tm['company_tier'] == 'Tier 3 (Long tail)', 'untimely_rate'].iloc[0]
untimely_t1 = tm.loc[tm['company_tier'] == 'Tier 1 (Top 10)',   'untimely_rate'].iloc[0]
relief_t1   = tm.loc[tm['company_tier'] == 'Tier 1 (Top 10)',   'any_relief'].iloc[0]
relief_t3   = tm.loc[tm['company_tier'] == 'Tier 3 (Long tail)', 'any_relief'].iloc[0]

fig = go.Figure()
fig.add_trace(go.Bar(name='Untimely',
    x=tiers, y=tm['untimely_rate'],
    marker=dict(color=CORAL, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['untimely_rate']],
    textposition='outside', textfont=dict(size=11, color=CORAL),
    hovertemplate='<b>%{x}</b><br>Untimely: <b>%{y:.1f}%</b><extra></extra>', cliponaxis=False,
))
fig.add_trace(go.Bar(name='Explanation only',
    x=tiers, y=tm['explanation_only'],
    marker=dict(color=NAVY, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['explanation_only']],
    textposition='outside', textfont=dict(size=11, color=NAVY),
    hovertemplate='<b>%{x}</b><br>Explanation only: <b>%{y:.1f}%</b><extra></extra>', cliponaxis=False,
))
fig.add_trace(go.Bar(name='Any relief',
    x=tiers, y=tm['any_relief'],
    marker=dict(color=SAGE, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>' for v in tm['any_relief']],
    textposition='outside', textfont=dict(size=11, color=SAGE),
    hovertemplate='<b>%{x}</b><br>Any relief: <b>%{y:.1f}%</b><extra></extra>', cliponaxis=False,
))

key_html = (
    f'<span style="color:{CORAL}">█</span> untimely   '
    f'<span style="color:{NAVY}">█</span> explanation only   '
    f'<span style="color:{SAGE}">█</span> any relief'
)
ymax = float(max(tm['explanation_only'])) * 1.22

fig.update_layout(
    barmode='group', bargap=0.32, bargroupgap=0.08,
    title=dict(
        text=(
            '<b style="font-size:20px;color:#0F172A">Long-tail companies are noticeably worse on every operational metric</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Outcome rates by company tier · Tier 3 is untimely <b>{untimely_t3:.1f}%</b> of the time vs '
            f'<b>{untimely_t1:.1f}%</b> for Tier 1, and delivers relief <b>{relief_t3:.1f}%</b> vs <b>{relief_t1:.1f}%</b>'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=540, margin=dict(l=60, r=40, t=150, b=100),
    xaxis=dict(title=dict(text='<i style="color:#94A3B8;font-size:10px">'
                               'Source: CFPB Consumer Complaint Database  ·  '
                               'Tier 1 = Top 10 companies by volume, Tier 2 = ranks 11-100, Tier 3 = all others</i>',
                          standoff=25),
               showgrid=False, showline=False, ticks='', tickfont=dict(size=12, color='#334155')),
    yaxis=dict(title='', range=[0, ymax], ticksuffix='%',
               gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8')),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)
fig.show()

**So what?**  A targeted compliance programme aimed at Tier 3 would lift the system-wide untimely 
rate well below 1%. Lifting Tier 3 relief rates by even 5 pp toward the Tier 1 average would 
translate to ~50,000 additional consumer remediations across the period. Mid-market players hoping 
to scale into Tier 2 can see exactly what quality gap they need to close.

## A5. Fast, but rarely remedial

77.5% of complaints close with explanation only. Only 5.8% deliver monetary relief, and 18.6% 
deliver any form of relief at all. The system is **fast** (97.5% timely) but the outcomes are **soft**.

In [19]:
resp = df['Company response to consumer'].value_counts().to_frame('count')
resp['pct'] = (resp['count'] / len(df) * 100).round(2)
resp

,count,pct
Company response to consumer,,
Closed with explanation,993221,77.45
Closed with non-monetary relief,158716,12.38
Closed with monetary relief,74243,5.79
Closed without relief,17868,1.39
Closed,17611,1.37
In progress,9277,0.72
Untimely response,6108,0.48
Closed with relief,5304,0.41
Unknown,7,0.00


In [20]:
resp_plot = resp['count'].iloc[::-1]
total_n = len(df)

def color_for(label):
    l = label.lower()
    if 'monetary' in l or label == 'Closed with relief':            return SAGE
    if label == 'Closed with non-monetary relief':                  return TEAL
    if label == 'Closed with explanation':                          return NAVY
    if 'without' in l or label == 'Untimely response':              return CORAL
    return '#94A3B8'

colors = [color_for(l) for l in resp_plot.index]
expl_pct = (df['Company response to consumer'] == 'Closed with explanation').mean() * 100
relief_pct = df['is_any_relief'].mean() * 100
xmax = resp_plot.max() * 1.32

fig = go.Figure()
fig.add_trace(go.Bar(
    y=list(resp_plot.index), x=list(resp_plot.values),
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v/1000:,.0f}K</b>   {v/total_n*100:.1f}%' for v in resp_plot.values],
    textposition='outside', textfont=dict(size=11, color='#334155'),
    customdata=[v/total_n*100 for v in resp_plot.values],
    hovertemplate='<b>%{y}</b><br>Count: %{x:,}<br>Share: %{customdata:.2f}%<extra></extra>',
    cliponaxis=False,
))

key_html = (
    f'<span style="color:{SAGE}">█</span> monetary/any relief   '
    f'<span style="color:{TEAL}">█</span> non-monetary relief   '
    f'<span style="color:{NAVY}">█</span> explanation only   '
    f'<span style="color:{CORAL}">█</span> no relief / untimely'
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Fast, but rarely remedial</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'How complaints close · {expl_pct:.1f}% close with explanation only, '
            f'only {relief_pct:.1f}% deliver any form of relief'
            '</span><br>'
            f'<span style="font-size:11px">{key_html}</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=560, margin=dict(l=270, r=80, t=150, b=90),
    xaxis=dict(title=dict(text=f'<i style="color:#94A3B8;font-size:10px">Source: CFPB Consumer Complaint Database  ·  '
                                f'n = {total_n:,} complaints  ·  Dec 2011 – May 2019</i>', standoff=25),
               range=[0, xmax], showgrid=True, gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10, color='#94A3B8'), tickformat=',.0f'),
    yaxis=dict(title='', showgrid=False, showline=False, ticks='',
               tickfont=dict(size=11, color='#334155'), automargin=True),
    showlegend=False, hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.30,
)
fig.show()

**So what?**  "Closed with explanation" looks like the cheap default — but the next insight 
shows that about 1 in 5 of those consumers come back disputing the resolution. That's not a 
free outcome.

There's also a missed-opportunity angle for ML: with 30% of complaints carrying a consumer 
narrative, a classifier could plausibly flag which "explanation-only" candidates are actually 
relief-eligible. A 1 pp shift toward relief is ~13K additional remediated consumers.

## A6. About 1 in 5 consumers dispute the resolution

**Caveat first:** dispute data only exists for the pre-April-2017 portion of the dataset 
(CFPB stopped collecting it). So we're working with ~770K rows here, not the full 1.28M.

Within that population, the average dispute rate is **19.3%**. Mortgage runs at 22.6%, 
Consumer Loan at 21.4%, Credit Card at 20.4% — meaning roughly 1 in 5 consumers who got an 
"explanation" came back unsatisfied.

In [21]:
disp = df[df['has_dispute_data'] == 1].groupby('Product').agg(
    n=('Complaint ID', 'count'),
    dispute_rate=('is_disputed', 'mean')
).reset_index()
disp = disp[disp['n'] >= 1000].sort_values('dispute_rate', ascending=False)
disp['dispute_rate_pct'] = (disp['dispute_rate'] * 100).round(2)
disp[['Product', 'n', 'dispute_rate_pct']]

,Product,n,dispute_rate_pct
7,Mortgage,226899,22.64
2,Consumer Loan,31605,21.42
3,Credit card,89190,20.41
8,Other financial service,1059,18.79
0,Bank account or service,86206,18.59
11,Student loan,32537,18.24
5,Debt collection,145835,17.58
4,Credit reporting,140432,15.75
6,Money transfers,5354,14.61
9,Payday loan,5544,14.38


In [22]:
avg_disp = df.loc[df['has_dispute_data'] == 1, 'is_disputed'].mean() * 100
disp_plot = disp.sort_values('dispute_rate_pct').tail(12).reset_index(drop=True)
colors = [CORAL if r > avg_disp else NAVY for r in disp_plot['dispute_rate_pct']]
short_labels = [p if len(p) < 42 else p[:39] + '…' for p in disp_plot['Product']]
xmax = disp_plot['dispute_rate_pct'].max() * 1.42

fig = go.Figure()
fig.add_vrect(x0=avg_disp, x1=xmax, fillcolor=CORAL, opacity=0.06, layer='below', line_width=0)
fig.add_trace(go.Bar(
    y=short_labels, x=disp_plot['dispute_rate_pct'],
    orientation='h',
    marker=dict(color=colors, line=dict(color='white', width=1.5)),
    text=[f'<b>{v:.1f}%</b>   n={n:,}' for v, n in zip(disp_plot['dispute_rate_pct'], disp_plot['n'])],
    textposition='outside', textfont=dict(size=11, color='#334155'),
    customdata=list(zip(disp_plot['Product'], disp_plot['n'])),
    hovertemplate='<b>%{customdata[0]}</b><br>Dispute rate: <b>%{x:.1f}%</b><br>Sample size: %{customdata[1]:,}<extra></extra>',
    cliponaxis=False,
))
fig.add_shape(type='line', xref='x', yref='paper',
              x0=avg_disp, x1=avg_disp, y0=0, y1=1,
              line=dict(color='#475569', width=1.5, dash='dash'))
fig.add_annotation(x=avg_disp, y=1.0, xref='x', yref='paper',
                   text=f'<b>avg {avg_disp:.1f}%</b>',
                   showarrow=False, yanchor='bottom', xanchor='center',
                   font=dict(color='#475569', size=11),
                   bgcolor='white', borderpad=3)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">About 1 in 5 consumers dispute the resolution</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Dispute rate by product · pre-April-2017 population (n = {df["has_dispute_data"].sum():,})  ·  '
            f'<span style="color:{CORAL}">█</span> above avg   '
            f'<span style="color:{NAVY}">█</span> below avg'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=620, margin=dict(l=300, r=80, t=110, b=110),
    xaxis=dict(title=dict(text='<i style="color:#94A3B8;font-size:10px">'
                               'Source: CFPB Consumer Complaint Database  ·  '
                               'Products with fewer than 1,000 complaints excluded  ·  '
                               'Dispute field discontinued by CFPB in April 2017</i>', standoff=25),
               range=[0, xmax], ticksuffix='%',
               gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8')),
    yaxis=dict(title='', showgrid=False, showline=False, ticks='',
               tickfont=dict(size=11, color='#334155'), automargin=True),
    showlegend=False, hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.32,
)
fig.show()

**So what?**  Mortgage portfolios should treat post-resolution dispute rate as a leading indicator 
of litigation exposure — it's both the largest product *and* the highest dispute rate. For 
consumer-loan and credit-card businesses, the gap to the population average represents thousands 
of preventable disputes over the period studied.

Operational caveat: since CFPB discontinued the field, any institution that wants this signal 
going forward has to recreate it internally — it's no longer in the public dataset.

## A7. Geographic propensity — per-capita tells a different story

Raw counts naturally favour the big states (CA, TX, FL). To see where complaints are *unusually* 
high, we normalise by population. Using 2017 Census estimates as the denominator — close enough 
to the dataset midpoint.

In [23]:
POP_2017 = {
    'CA':39536653,'TX':28304596,'FL':20984400,'NY':19849399,'PA':12805537,'IL':12802023,'OH':11658609,
    'GA':10429379,'NC':10273419,'MI':9962311,'NJ':9005644,'VA':8470020,'WA':7405743,'AZ':7016270,
    'MA':6859819,'TN':6715984,'IN':6666818,'MO':6113532,'MD':6052177,'WI':5795483,'CO':5607154,
    'MN':5576606,'SC':5024369,'AL':4874747,'LA':4684333,'KY':4454189,'OR':4142776,'OK':3930864,
    'CT':3588184,'UT':3101833,'IA':3145711,'NV':2998039,'AR':3004279,'MS':2984100,'KS':2913123,
    'NM':2088070,'NE':1920076,'WV':1815857,'ID':1716943,'HI':1427538,'NH':1342795,'ME':1335907,
    'MT':1050493,'RI':1059639,'DE':961939,'SD':869666,'ND':755393,'AK':739795,'DC':693972,
    'VT':623657,'WY':579315,'PR':3337177,
}

st = df.loc[df['State'] != 'Unknown', 'State'].value_counts().reset_index()
st.columns = ['State', 'complaints']
st['pop'] = st['State'].map(POP_2017)
st = st.dropna()
st['per_100k'] = (st['complaints'] / st['pop'] * 100000).round(1)

print("Top 15 per 100k residents:")
st.sort_values('per_100k', ascending=False).head(15)

Top 15 per 100k residents:


,State,complaints,pop,per_100k
32,DC,6886,693972.0,992.3
4,GA,67102,10429379.0,643.4
34,DE,6150,961939.0,639.3
11,MD,36545,6052177.0,603.8
1,FL,126487,20984400.0,602.8
20,NV,16006,2998039.0,533.9
6,NJ,47799,9005644.0,530.8
0,CA,176009,39536653.0,445.2
10,VA,37471,8470020.0,442.4
3,NY,86143,19849399.0,434.0


In [24]:
# Side-by-side: absolute and per-capita, for the top 15 of each
top_abs = st.sort_values('complaints', ascending=False).head(15).sort_values('complaints')
top_pc  = st.sort_values('per_100k',   ascending=False).head(15).sort_values('per_100k')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('<b>Top 15 by absolute count</b>',
                                    '<b>Top 15 per 100,000 residents</b>'),
                    horizontal_spacing=0.16)

fig.add_trace(go.Bar(
    x=top_abs['complaints'], y=top_abs['State'], orientation='h',
    text=[f'{v/1000:,.0f}K' for v in top_abs['complaints']], textposition='outside',
    textfont=dict(size=10, color='#334155'),
    marker=dict(color=NAVY, line=dict(color='white', width=1.2)),
    hovertemplate='<b>%{y}</b><br>Complaints: %{x:,}<extra></extra>',
    cliponaxis=False, showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=top_pc['per_100k'], y=top_pc['State'], orientation='h',
    text=[f'{v:.0f}' for v in top_pc['per_100k']], textposition='outside',
    textfont=dict(size=10, color='#334155'),
    marker=dict(color=CORAL, line=dict(color='white', width=1.2)),
    hovertemplate='<b>%{y}</b><br>Per 100k: %{x:.1f}<extra></extra>',
    cliponaxis=False, showlegend=False,
), row=1, col=2)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Geography: absolute vs per-capita tell different stories</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            'Population favours CA/TX/FL on absolute count — DC/DE/MD lead on propensity to complain'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=580, margin=dict(l=60, r=70, t=120, b=80),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.32,
)
fig.update_xaxes(showgrid=True, gridcolor='#EEF2F6', griddash='dot', showline=False, ticks='',
                 tickfont=dict(size=10, color='#94A3B8'))
fig.update_yaxes(showgrid=False, showline=False, ticks='',
                 tickfont=dict(size=11, color='#334155'), automargin=True)
fig.show()

**So what?**  DC, Delaware, and Maryland top the per-capita list — all small jurisdictions with 
heavy financial-sector concentration and strong consumer-protection awareness. That's partly 
genuine financial-services density and partly a "complaint-savvy population" effect.

The interesting cells are **Florida, Georgia, and Nevada** — they show up high on *both* 
per-capita and absolute counts. That combination (real volume + real propensity to complain) 
makes them the highest signal-to-noise targets for a compliance or CX investment.

(Note: an interactive choropleth would also work nicely here, but Plotly's `USA-states` choropleth 
needs a CDN-hosted topojson; the dual-bar view above carries the same information without that 
dependency.)

---

# Part B — Drill-down on the 3 credit bureaus

Part A showed the credit bureaus are ~25% of all complaints. They also have something the other 
top-10 companies don't — **they all do the same job**. Equifax, Experian and TransUnion are 
direct competitors providing nearly identical services. That makes them genuinely comparable 
in a way that, say, a bank and a debt-collector aren't.

So this part of the analysis treats the three of them as a peer group and builds an accountability 
scorecard. We need a slightly tighter subset and one extra cleaning step on the issue taxonomy — 
covered next.

## B0. Subset and taxonomy normalisation

In [25]:
df_b = df[df['Company'].isin(BUREAU_MAP.keys())].copy()
print(f"Bureau subset shape: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns")
print(f"Share of full dataset: {len(df_b)/len(df)*100:.1f}%")
print()
print("Bureau breakdown:")
print(df_b['bureau'].value_counts())

Bureau subset shape: 316,074 rows × 31 columns
Share of full dataset: 24.6%

Bureau breakdown:
bureau
Equifax       115703
Experian      103784
TransUnion     96587
Name: count, dtype: int64


The CFPB renamed several issue categories over the dataset window. Without normalisation we'd 
double-count the same underlying issue under both its old and new label. We verified each pair 
by checking date-range overlap in the original cleaning — the renames are merged here.

In [26]:
df_b['Issue'] = df_b['Issue'].replace({
    'Incorrect information on credit report':
        'Incorrect information on your report',
    "Credit reporting company's investigation":
        "Problem with a credit reporting company's investigation into an existing problem",
    'Improper use of my credit report':
        'Improper use of your report',
    'Unable to get credit report/credit score':
        'Unable to get your credit report or credit score',
    'Credit monitoring or identity protection':
        'Credit monitoring or identity theft protection services',
})
print("Top issues after normalisation:")
df_b['Issue'].value_counts().head(8)

Top issues after normalisation:


Issue
Incorrect information on your report                                                199319
Problem with a credit reporting company's investigation into an existing problem     57107
Improper use of your report                                                          27495
Unable to get your credit report or credit score                                     15034
Credit monitoring or identity theft protection services                               6280
Problem with fraud alerts or security freezes                                         4663
Attempts to collect debt not owed                                                     3934
Written notification about debt                                                        427
Name: count, dtype: int64

## B1. Bureau volume and top issues

Two views on the same page:
- which bureau attracts the most complaints
- what those complaints are actually about

In [27]:
ISSUE_LABELS = {
    'Incorrect information on your report': 'Incorrect information',
    "Problem with a credit reporting company's investigation into an existing problem": 'Investigation problem',
    'Improper use of your report': 'Improper use of report',
    'Unable to get your credit report or credit score': 'Unable to get report/score',
    'Credit monitoring or identity theft protection services': 'Credit monitoring',
    'Problem with fraud alerts or security freezes': 'Fraud alerts / freezes',
    'Attempts to collect debt not owed': 'Debt not owed',
}

# Bureau volume
vol = df_b['bureau'].value_counts().reindex(['Equifax','Experian','TransUnion'])
# Top issues
top_issues = df_b['Issue'].value_counts().head(7).reset_index()
top_issues.columns = ['issue', 'count']
top_issues['short'] = top_issues['issue'].replace(ISSUE_LABELS)
top_issues = top_issues.sort_values('count')

inc_pct = (df_b['Issue'] == 'Incorrect information on your report').mean() * 100

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('<b>Complaint volume by bureau</b>', '<b>Top issues</b>'),
                    column_widths=[0.40, 0.60], horizontal_spacing=0.18)

# left: bureau volume
fig.add_trace(go.Bar(
    x=list(vol.index), y=list(vol.values),
    marker=dict(color=[BUREAU_COLORS[b] for b in vol.index], line=dict(color='white', width=1.5)),
    text=[f'<b>{v/1000:,.0f}K</b>' for v in vol.values], textposition='outside',
    textfont=dict(size=13, color='#334155'),
    hovertemplate='<b>%{x}</b><br>Complaints: %{y:,}<extra></extra>',
    cliponaxis=False, showlegend=False,
), row=1, col=1)

# right: top issues
fig.add_trace(go.Bar(
    y=top_issues['short'], x=top_issues['count'], orientation='h',
    marker=dict(color=NAVY, line=dict(color='white', width=1.2)),
    text=[f'<b>{v/1000:,.0f}K</b>   {v/len(df_b)*100:.0f}%' for v in top_issues['count']],
    textposition='outside', textfont=dict(size=10, color='#334155'),
    hovertemplate='<b>%{y}</b><br>Complaints: %{x:,}<extra></extra>',
    cliponaxis=False, showlegend=False,
), row=1, col=2)

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">'
            f'Equifax leads volume; {inc_pct:.0f}% of all bureau complaints are about wrong data on the report'
            '</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'Volume left, issue mix right · bureau subset n = {len(df_b):,}'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=540, margin=dict(l=60, r=80, t=110, b=80),
    bargap=0.32,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
)
fig.update_xaxes(showgrid=True, gridcolor='#EEF2F6', griddash='dot', showline=False, ticks='',
                 tickfont=dict(size=10, color='#94A3B8'))
fig.update_yaxes(showgrid=False, showline=False, ticks='',
                 tickfont=dict(size=11, color='#334155'), automargin=True)
# turn off gridlines on the categorical x-axis of the left panel
fig.update_xaxes(showgrid=False, row=1, col=1, tickfont=dict(size=12, color='#334155'))
fig.update_yaxes(range=[0, max(vol.values)*1.18], row=1, col=1, tickformat=',d')
fig.show()

**So what?**  Equifax draws 19.8% more complaints than TransUnion — the highest among the three. 
More importantly, **63% of all bureau complaints (199,319) are about incorrect information on the 
report** — the exact data banks use for underwriting every loan. If a bureau fails to fix that 
data, the downstream cost is wrong lending decisions, not just consumer frustration.

## B2. Speed vs quality — Experian's 2.6× edge on actually fixing things

All three bureaus respond on time (≥98.6%). But responding on time and fixing the problem are 
not the same thing. Experian closes 39.7% of complaints with non-monetary relief — nearly 3× 
the rate of Equifax (15.7%) and TransUnion (14.5%).

In [28]:
timely = df_b.groupby('bureau')['Timely response?'].value_counts(normalize=True).unstack() * 100
timely = timely.reindex(['Equifax','Experian','TransUnion'])
print("Timely-response breakdown:")
print(timely.round(2))
print()
resolution = df_b.groupby('bureau')['Company response to consumer'].value_counts(normalize=True).unstack() * 100
resolution = resolution.reindex(['Equifax','Experian','TransUnion'])
print("Resolution mix:")
print(resolution[['Closed with explanation','Closed with non-monetary relief','Closed with monetary relief']].round(1))

Timely-response breakdown:
Timely response?    No    Yes
bureau                       
Equifax           1.41  98.59
Experian          0.01  99.99
TransUnion        0.09  99.91

Resolution mix:
Company response to consumer  Closed with explanation  Closed with non-monetary relief  Closed with monetary relief
bureau                                                                                                             
Equifax                                          80.3                             15.7                          0.0
Experian                                         58.0                             39.7                          0.7
TransUnion                                       85.2                             14.5                          0.3


In [29]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Untimely response rate</b>', '<b>Resolution mix (% closure type)</b>'),
    column_widths=[0.32, 0.68], horizontal_spacing=0.14,
)

# Panel 1 — untimely %
untimely = timely['No'] if 'No' in timely.columns else pd.Series([0,0,0], index=timely.index)
fig.add_trace(go.Bar(
    x=untimely.index, y=untimely.values,
    text=[f'<b>{v:.2f}%</b>' for v in untimely.values],
    textposition='outside', textfont=dict(size=11, color='#334155'),
    marker=dict(color=[BUREAU_COLORS[b] for b in untimely.index], line=dict(color='white', width=1.5)),
    hovertemplate='<b>%{x}</b><br>Untimely: %{y:.2f}%<extra></extra>',
    cliponaxis=False, showlegend=False,
), row=1, col=1)

# Panel 2 — stacked resolution mix
res_cols = ['Closed with explanation', 'Closed with non-monetary relief', 'Closed with monetary relief']
res_colors = {'Closed with explanation': CORAL,
              'Closed with non-monetary relief': SAGE,
              'Closed with monetary relief': TEAL}
for col in res_cols:
    if col in resolution.columns:
        vals = resolution[col].values
        fig.add_trace(go.Bar(
            x=resolution.index, y=vals, name=col,
            marker=dict(color=res_colors[col], line=dict(color='white', width=1.2)),
            text=[f'<b>{v:.1f}%</b>' for v in vals],
            textposition='inside', textfont=dict(color='white', size=11),
            hovertemplate=f'<b>{col}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>',
        ), row=1, col=2)

fig.update_layout(
    barmode='stack',
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Speed vs quality — Experian acts on 39.7% of cases vs ~15% for the others</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            'All three bureaus reply on time ≥98.6%; the differentiator is resolution outcome'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=540, margin=dict(l=60, r=40, t=110, b=110),
    legend=dict(orientation='h', yanchor='top', y=-0.10, xanchor='center', x=0.5,
                font=dict(size=11, color='#475569'), bgcolor='rgba(0,0,0,0)'),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.32,
)
fig.update_xaxes(showgrid=False, showline=False, ticks='', tickfont=dict(size=12, color='#334155'))
fig.update_yaxes(showgrid=True, gridcolor='#EEF2F6', griddash='dot', showline=False, ticks='', zeroline=False,
                 tickfont=dict(size=10.5, color='#94A3B8'), ticksuffix='%')
fig.update_yaxes(range=[0, 1.7], row=1, col=1)  # untimely panel zoom
fig.show()

**So what?**  Banks relying on Equifax or TransUnion for credit-quality-sensitive decisions are 
working with bureaus that rarely correct errors — the wrong data stays on credit reports longer. 
The 2.6× gap on resolution rate is the single most actionable finding in Part B.

Caveat: a higher resolution rate could reflect a higher willingness to correct errors **or** 
simply a higher rate of errors needing correction. Disambiguating those needs internal bureau 
data we don't have. Either way, from the consumer's point of view, Experian closes their case 
with action far more often.

## B3. The Equifax breach monthly signature

Yearly aggregation shows that Equifax complaints jumped 92% from 2016 to 2017. Monthly 
aggregation shows the *signature*: the spike begins August 2017 and peaks in September — the 
same month Equifax disclosed the breach affecting 147 million Americans. Experian and TransUnion 
show no comparable spike in the same window.

In [30]:
df_b['year_month_dt'] = df_b['Date received'].dt.to_period('M').dt.to_timestamp()
trend_m = df_b.groupby(['year_month_dt','bureau']).size().reset_index(name='n')

fig = px.line(trend_m, x='year_month_dt', y='n', color='bureau',
              color_discrete_map=BUREAU_COLORS)
fig.update_traces(line=dict(width=2.2))

# Identify Equifax peak month for the annotation
eq_peak = trend_m[trend_m['bureau']=='Equifax'].sort_values('n', ascending=False).iloc[0]
fig.add_annotation(
    x=eq_peak['year_month_dt'], y=eq_peak['n'],
    text=f"<b>Equifax breach</b><br>{eq_peak['year_month_dt'].strftime('%b %Y')}  |  {int(eq_peak['n']):,}",
    showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=1.5, arrowcolor=CORAL,
    ax=70, ay=-60, font=dict(color=CORAL, size=11),
    bgcolor='rgba(255,255,255,0.95)', bordercolor=CORAL, borderwidth=1.2, borderpad=6, align='left',
)
fig.add_vline(x='2017-09-07', line_dash='dot', line_color='#94A3B8', line_width=1.5,
              annotation_text='breach disclosed<br>Sept 7, 2017',
              annotation_position='top right',
              annotation_font=dict(color='#94A3B8', size=10))

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Monthly granularity reveals the Equifax breach signature</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            'Complaint volume by bureau, monthly · the spike is uniquely Equifax-driven'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=520, margin=dict(l=70, r=40, t=110, b=80),
    legend=dict(orientation='h', yanchor='top', y=-0.10, xanchor='center', x=0.5,
                title='', font=dict(size=11, color='#475569'), bgcolor='rgba(0,0,0,0)'),
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    xaxis=dict(title='', showgrid=False, showline=False, ticks='',
               tickfont=dict(size=11, color='#334155')),
    yaxis=dict(title='Complaints', gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8'), tickformat=',d'),
)
fig.show()

**So what?**  An anomaly-detection model on monthly bureau volume would have flagged the August 
2017 rise — three weeks before the public disclosure on September 7. Real-time monitoring of 
complaint volume against a single bureau is a leading indicator of systemic data-quality risk. 
Any bank relying solely on Equifax during this window was exposed.

## B4. Consumer dispute rate by bureau

Among the pre-2017 subset where dispute outcomes were recorded, Equifax's dispute rate is 20.8% 
vs Experian's 11.7% — a 78% relative gap.

In [31]:
disp_b = df_b[df_b['has_dispute_data']==1]
dispute_rate = (disp_b.groupby('bureau')['is_disputed'].mean() * 100).reindex(['Equifax','Experian','TransUnion'])
n_per = disp_b.groupby('bureau').size().reindex(['Equifax','Experian','TransUnion'])

print("Dispute rate by bureau (pre-April-2017 reported population):")
pd.DataFrame({'n': n_per, 'dispute_rate_pct': dispute_rate.round(2)})

Dispute rate by bureau (pre-April-2017 reported population):


,n,dispute_rate_pct
bureau,,
Equifax,48390,20.83
Experian,45622,11.68
TransUnion,40006,14.09


In [32]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(dispute_rate.index), y=list(dispute_rate.values),
    text=[f'<b>{v:.1f}%</b><br><span style="font-size:10px;color:#94A3B8">n={n:,}</span>'
          for v, n in zip(dispute_rate.values, n_per.values)],
    textposition='outside', textfont=dict(size=13, color='#334155'),
    marker=dict(color=[BUREAU_COLORS[b] for b in dispute_rate.index], line=dict(color='white', width=1.5)),
    hovertemplate='<b>%{x}</b><br>Dispute rate: %{y:.2f}%<extra></extra>',
    cliponaxis=False,
))

avg = dispute_rate.mean()
fig.add_hline(y=avg, line_dash='dash', line_color='#475569', line_width=1.5,
              annotation_text=f'avg {avg:.1f}%',
              annotation_position='top right',
              annotation_font=dict(color='#475569', size=11))

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#0F172A">Equifax draws 78% more consumer disputes than Experian</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            f'% of resolved complaints later disputed by the consumer · '
            f'pre-April-2017 reported population (n = {n_per.sum():,})'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=480, margin=dict(l=70, r=40, t=110, b=80),
    showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='#CBD5E1', font_size=12),
    bargap=0.45,
    xaxis=dict(showgrid=False, showline=False, ticks='', tickfont=dict(size=13, color='#334155')),
    yaxis=dict(title='', range=[0, max(dispute_rate.values)*1.25], ticksuffix='%',
               gridcolor='#EEF2F6', griddash='dot',
               showline=False, ticks='', zeroline=False,
               tickfont=dict(size=10.5, color='#94A3B8')),
)
fig.show()

**So what?**  Dispute rate is the *lagging indicator of resolution quality* and the 
*leading indicator of legal exposure*. The pattern aligns with B2 — the bureau closing most 
cases with explanation-only also draws the most pushback. A closed-with-explanation complaint 
that is later disputed is the same complaint that shows up in a class-action ten months later.

## B5. Issue fingerprint — each bureau's complaint mix is different

Are the bureaus uniformly bad, or each bad at *different* things? We compute each bureau's 
share of complaints across the top 10 issues — a fingerprint of where their volume concentrates.

In [33]:
top10 = df_b['Issue'].value_counts().head(10).index.tolist()
finger = df_b[df_b['Issue'].isin(top10)].groupby(['bureau','Issue']).size().unstack(fill_value=0)
finger = finger.div(finger.sum(axis=1), axis=0) * 100
finger = finger.reindex(['Equifax','Experian','TransUnion'])
finger.columns = [ISSUE_LABELS.get(c, (c[:32]+'…' if len(c)>32 else c)) for c in finger.columns]
finger.round(1)

,Debt not owed,Credit monitoring,False statements or representati…,Identity theft protection or oth…,Improper use of report,Incorrect information,Investigation problem,Fraud alerts / freezes,Unable to get report/score,Written notification about debt
bureau,,,,,,,,,,
Equifax,1.1,2.0,0.1,0.1,11.3,60.2,17.8,1.7,5.6,0.1
Experian,1.3,2.3,0.1,0.0,6.5,64.7,18.8,1.6,4.7,0.1
TransUnion,1.4,1.7,0.1,0.0,8.1,65.7,17.9,1.1,3.9,0.1


In [34]:
fig = px.imshow(
    finger.values,
    x=finger.columns, y=finger.index,
    color_continuous_scale='Reds', aspect='auto',
    text_auto='.1f',
    labels={'color': '% of bureau'},
)
fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Issue fingerprint — same top issue, different mix beneath</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            '% of each bureau\'s complaints falling under the top 10 issue categories'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=440, margin=dict(l=130, r=80, t=110, b=160),
    coloraxis_colorbar=dict(thickness=14, len=0.75, ticksuffix='%',
                            outlinewidth=0, tickfont=dict(size=10, color='#94A3B8')),
    xaxis=dict(tickangle=-25, tickfont=dict(size=10, color='#334155'),
               showgrid=False, showline=False, ticks=''),
    yaxis=dict(tickfont=dict(size=12, color='#334155'),
               showgrid=False, showline=False, ticks='', automargin=True),
)
fig.show()

**So what?**  All three bureaus are dominated by *Incorrect information* (60–66%), so the systemic 
problem is shared. But the secondary mix diverges meaningfully — investigation problems, improper 
use, and credit-monitoring issues all have visibly different shares per bureau. Risk teams should 
not treat the three as interchangeable; the issue-fingerprint is a one-page artefact for any 
bureau-procurement conversation.

## B6. Resolution flow — the whole story in one diagram

Sankey diagram of bureau → top issue → resolution outcome. The visual makes the asymmetry 
obvious at a glance: most volume flows into *Incorrect information*, and most of that flows 
into *Closed with explanation*.

In [35]:
top4_iss = df_b['Issue'].value_counts().head(4).index.tolist()
top4_resp = df_b['Company response to consumer'].value_counts().head(4).index.tolist()
flow = df_b[df_b['Issue'].isin(top4_iss) & df_b['Company response to consumer'].isin(top4_resp)].copy()
flow['issue_short'] = flow['Issue'].replace(ISSUE_LABELS)

bureaus = ['Equifax','Experian','TransUnion']
issues  = sorted(flow['issue_short'].unique().tolist())
resps   = sorted(flow['Company response to consumer'].unique().tolist())
labels  = bureaus + issues + resps
idx     = {n: i for i, n in enumerate(labels)}

# Bureau → Issue links
b2i = flow.groupby(['bureau','issue_short']).size().reset_index(name='v')
# Issue → Outcome links
i2r = flow.groupby(['issue_short','Company response to consumer']).size().reset_index(name='v')

bcr = {'Equifax':'rgba(231,76,60,0.40)',
       'Experian':'rgba(52,152,219,0.40)',
       'TransUnion':'rgba(46,204,113,0.40)'}

src, tgt, val, lc = [], [], [], []
for _, r in b2i.iterrows():
    src.append(idx[r['bureau']]); tgt.append(idx[r['issue_short']]); val.append(int(r['v']))
    lc.append(bcr[r['bureau']])
for _, r in i2r.iterrows():
    src.append(idx[r['issue_short']]); tgt.append(idx[r['Company response to consumer']]); val.append(int(r['v']))
    lc.append('rgba(150,150,150,0.30)')

node_colors = ([BUREAU_COLORS[b] for b in bureaus]
               + ['#34495e'] * len(issues)
               + ['#7f8c8d'] * len(resps))

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(label=labels, color=node_colors, pad=18, thickness=20,
              line=dict(color='white', width=0.5)),
    link=dict(source=src, target=tgt, value=val, color=lc),
))
fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">Resolution flow: most volume ends as explanation, not correction</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            'Bureau → top 4 issues → top 4 outcomes · width = number of complaints'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=600, margin=dict(l=20, r=20, t=110, b=40),
    font=dict(size=12),
)
fig.show()

**So what?**  The Sankey is the executive summary of the whole accountability story. Two visible 
patterns: (1) every bureau's volume converges on *Incorrect information*, and (2) the dominant 
downstream flow is into *Closed with explanation* — the no-action outcome. Experian's flow into 
*Closed with non-monetary relief* is visibly thicker than the others'.

**That is the gap the industry needs to close.**

## B7. The 7-dimension accountability scorecard

Pulling it all together. Each bureau is scored on seven dimensions — five from the original 
scorecard plus two derived in this notebook (response lag, dispute rate). Each dimension is 
rescaled to 0–100 (100 = best of the three, 20 = worst). The radar shape compares the three 
across all dimensions at once; the final score is the mean.

| Dimension | What it measures | Better when |
|---|---|---|
| Complaint volume | Total complaints received | Lower |
| Timely response | % responding within 15 days | Higher |
| Resolution quality | % closed with non-monetary relief | Higher |
| Trend | Volume growth 2016 → 2018 | Lower |
| Vulnerable groups | Resolution quality for tagged groups | Higher |
| Response lag | Mean lag in days from received to sent | Lower |
| Dispute rate | % of resolved complaints later disputed | Lower |

In [36]:
def rescale(values, invert=False, lo=20, hi=100):
    v = np.array(values, dtype=float)
    if invert:
        v = -v
    rng = v.max() - v.min()
    if rng == 0:
        return np.full_like(v, (lo + hi) / 2)
    return lo + ((v - v.min()) / rng) * (hi - lo)

bureaus_order = ['Equifax','Experian','TransUnion']

# 1. Volume (lower = better)
vols = df_b['bureau'].value_counts().reindex(bureaus_order).values

# 2. Timely % (higher = better)
timely_pct = (df_b.groupby('bureau').apply(
    lambda x: (x['Timely response?']=='Yes').mean()*100, include_groups=False
).reindex(bureaus_order).values)

# 3. Resolution quality - % non-monetary relief (higher = better)
res_q = (df_b.groupby('bureau').apply(
    lambda x: (x['Company response to consumer']=='Closed with non-monetary relief').mean()*100,
    include_groups=False
).reindex(bureaus_order).values)

# 4. Trend - 2018 / 2016 ratio (lower = better)
trend = df_b.groupby(['bureau','year']).size().unstack(fill_value=0)
trend_ratio = (trend.get(2018, 0) / trend.get(2016, 1).replace(0, 1)).reindex(bureaus_order).values

# 5. Vulnerable groups - any-relief rate for tagged consumers (higher = better)
tagged = df_b[df_b['Tags'].notna()]
vuln = (tagged.groupby('bureau')['is_any_relief'].mean()*100).reindex(bureaus_order).fillna(0).values

# 6. Response lag - mean days (lower = better)
lag_clean = df_b[(df_b['lag_days']>=0)&(df_b['lag_days']<=30)]
lag_mean = lag_clean.groupby('bureau')['lag_days'].mean().reindex(bureaus_order).values

# 7. Dispute rate (lower = better)
disp_rate = (disp_b.groupby('bureau')['is_disputed'].mean()*100).reindex(bureaus_order).values

scorecard = pd.DataFrame({
    'bureau':           bureaus_order,
    'volume_score':     rescale(vols, invert=True),
    'timely_score':     rescale(timely_pct),
    'resolution_score': rescale(res_q),
    'trend_score':      rescale(trend_ratio, invert=True),
    'vulnerable_score': rescale(vuln),
    'lag_score':        rescale(lag_mean, invert=True),
    'dispute_score':    rescale(disp_rate, invert=True),
})
scorecard['final_score'] = scorecard.drop(columns='bureau').mean(axis=1)
scorecard.round(1)

,bureau,volume_score,timely_score,resolution_score,trend_score,vulnerable_score,lag_score,dispute_score,final_score
0,Equifax,20.0,20.0,23.9,100.0,21.3,100.0,20.0,43.6
1,Experian,69.9,100.0,100.0,30.2,100.0,20.0,100.0,74.3
2,TransUnion,100.0,95.1,20.0,20.0,20.0,20.3,78.9,50.6


In [37]:
cats = ['Complaint<br>volume', 'Timely<br>response', 'Resolution<br>quality',
        'Trend', 'Vulnerable<br>groups', 'Response<br>lag', 'Dispute<br>rate']

fig = go.Figure()
for bureau in bureaus_order:
    row = scorecard[scorecard['bureau']==bureau].iloc[0]
    vals = [row['volume_score'], row['timely_score'], row['resolution_score'],
            row['trend_score'], row['vulnerable_score'],
            row['lag_score'], row['dispute_score']]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=cats + [cats[0]],
        fill='toself',
        name=f"{bureau} ({row['final_score']:.1f})",
        line=dict(color=BUREAU_COLORS[bureau], width=2.5),
        opacity=0.85,
        hovertemplate='<b>'+bureau+'</b><br>%{theta}: %{r:.1f}<extra></extra>',
    ))

ranking_text = "<br>".join(
    f"<b style='color:{BUREAU_COLORS[b]}'>{i+1}. {b}: {scorecard.loc[scorecard['bureau']==b,'final_score'].iloc[0]:.1f} / 100</b>"
    for i, b in enumerate(scorecard.sort_values('final_score', ascending=False)['bureau'])
)

fig.update_layout(
    polar=dict(
        bgcolor='white',
        radialaxis=dict(visible=True, range=[0, 100],
                        showticklabels=True, tickfont=dict(size=9, color='#94A3B8'),
                        gridcolor='#E2E8F0'),
        angularaxis=dict(tickfont=dict(size=11, color='#334155'),
                         linecolor='#E2E8F0'),
    ),
    title=dict(
        text=(
            '<b style="font-size:21px;color:#0F172A">7-dimension Bureau Accountability Scorecard</b><br>'
            '<span style="font-size:13px;color:#64748B">'
            'Higher is better · each dimension rescaled to 0–100 across the three bureaus'
            '</span>'
        ),
        x=0.02, xanchor='left', y=0.97,
    ),
    paper_bgcolor='white',
    height=640, margin=dict(t=110, b=60, l=60, r=60),
    legend=dict(orientation='h', yanchor='bottom', y=-0.10, xanchor='center', x=0.5,
                title='', font=dict(size=12, color='#334155'), bgcolor='rgba(0,0,0,0)'),
    annotations=[dict(
        xref='paper', yref='paper', x=1.0, y=1.05, xanchor='right', yanchor='top',
        text=ranking_text, showarrow=False, align='right',
        font=dict(size=11),
    )],
)
fig.show()

**Final ranking**

- **Experian: 74.3 / 100.** Leads on resolution quality, timeliness, dispute rate, and 
  vulnerable-group relief outcomes. Trails only on trend (it grew most over the period) and 
  routing lag.
- **TransUnion: 50.6 / 100.** Lowest absolute complaint volume and fast timely response, 
  but weakest on resolution quality and slowest on trend recovery.
- **Equifax: 43.6 / 100.** Best on trend recovery (the post-breach normalisation pulled it 
  back) and fastest routing lag — but weakest on the metrics that matter for consumer 
  outcomes (resolution, dispute, vulnerable groups).

**So what?**  No bureau dominates all seven dimensions — but Experian dominates the ones that 
matter most for *resolution outcomes*. A diversified bureau strategy with Experian as the anchor 
for quality-sensitive decisions, and ongoing monitoring of Equifax's complaint trajectory and 
post-resolution dispute rate, follows directly from this picture.

---

# Part C — Synthesis

## C1. Limitations & validation

What this dataset can and cannot tell us, and the checks we ran to ground the findings.

| Limitation | What it means | What we did about it |
|---|---|---|
| Self-selected dataset | Not every consumer issue becomes a CFPB complaint | Treat metrics as *complaint propensity*, not absolute consumer harm |
| Dispute field discontinued (Apr 2017) | 40% of rows have no dispute value | Compute dispute rates only on the pre-2017 population; disclose sample size on every chart |
| No customer identifier | Can't do RFM- or LTV-style analysis | Use company-tier segmentation as the closest proxy |
| No financial-impact field | Can't compute dollar value of relief | Use relief-rate as a proxy; flag as enrichment opportunity |
| Single-machine pandas | 1.6 GB working set is near the comfort limit | Production version would need Spark / Databricks |
| Taxonomy drift over the period | "Credit reporting" was renamed mid-2017 | Treat both old and new labels; acknowledge the regime shift |
| Vulnerable-group sample sizes | Some tag groups have < 2K rows | Reported with sample sizes; small groups are descriptive, not statistically robust |

**Validation checks we ran:**

- Total complaint counts reconcile to CFPB published yearly totals (within rounding)
- Top-company list cross-checks with CFPB annual reports
- The September 2017 spike aligns with the publicly documented Equifax breach announcement 
  (Sept 7, 2017) — a strong external anchor for the time series
- The post-2017 disappearance of the dispute field cross-checks with CFPB's published policy change

## C2. Recommendations

### Strategy

1. **Make credit-reporting accuracy the priority category.** It has overtaken mortgage as the 
   dominant complaint driver — and it is still growing.
2. **Anchor a diversified bureau strategy on Experian.** Its 2.6× resolution-quality edge is 
   the most reliable signal in the bureau data. Use it for credit-quality-sensitive use cases.
3. **Targeted Tier-3 supervision programme.** Long-tail companies miss SLAs at roughly 10× 
   the rate of Top-100s; a focused compliance push there would pull system-wide untimely rate 
   below 1%.

### Operations

4. **Resolution-quality scorecard, not just response-time scorecard.** Track relief-rate at 
   the company level alongside timeliness. The system is fast (97.5% timely) but soft — only 
   5.8% deliver monetary relief.
5. **Dispute-risk model on the explanation-only population.** Use product, issue, state, and 
   company features to flag closures most likely to generate a consumer dispute. A 10% reduction 
   = ~15,000 fewer adverse events over a comparable period.
6. **Real-time anomaly monitoring on bureau volume.** An anomaly-detection layer on monthly 
   bureau volume would have flagged the Equifax breach 3 weeks before public disclosure. 
   That is leading-indicator value worth investing in.

### Data engineering

7. **Move product/issue taxonomy to a dimension table** so renames (like "Credit reporting" → 
   "Credit reporting, credit repair services, …") don't keep breaking downstream metrics.
8. **Migrate to Spark or Databricks** before the dataset doubles. 1.6 GB in pandas is already 
   a fragile foundation for anything running on a schedule.

### Customer experience

9. **NLP triage on the ~30% of complaints with narratives.** Classify by severity and route 
   high-risk cases to a remediation team instead of defaulting to explanation-only.

## C3. Roadmap

**Short-term (0–3 months)**

- Automate ETL from the CFPB public API → cleaned warehouse table (daily incremental)
- Build a BI dashboard (Power BI / Tableau / Superset) for Compliance and CX teams
- Standardise company-name fuzzy matching to collapse 5,275 raw names into a clean entity table
- Stand up the 7-dimension scorecard as a monthly automated artefact

**Medium-term (3–12 months)**

- Train and deploy a dispute-risk classifier on the explanation-only population
- Migrate from CSV + single-machine pandas to a Spark / Databricks pipeline
- Add NLP topic models on the consented narratives
- Real-time anomaly detection on monthly bureau volume

**Long-term (12+ months)**

- Near-real-time intake monitoring — catch Equifax-style spikes in hours, not months
- Federated benchmarking product for member institutions
- ML-driven resolution-quality scoring at company × product × geography
- LLM-assisted compliance review with closed-domain RAG, citation-required outputs and full audit logging

## C4. Closing

We took ~1.28 million rows of regulatory-grade complaint data, cleaned it, engineered the 
operational and risk features needed, and pulled out a coherent story across both the broad 
market and the three credit bureaus:

- Concentration is extreme (top 10 = 52% of volume, three bureaus alone = 25%)
- Product mix has flipped (mortgage out, credit reporting in)
- Channel went almost entirely digital
- The system is fast (97.5% timely) but soft (5.8% monetary relief, 19.3% dispute)
- Among the three bureaus, Experian leads decisively on resolution quality (74.3 / 100 on the 
  composite scorecard, vs TransUnion 50.6 and Equifax 43.6)

> **One-sentence takeaway:** complaint volume in U.S. financial services is highly concentrated, 
> increasingly digital, dominated by credit-reporting issues, and resolved fast but rarely 
> remediated — making **resolution quality** the single highest-ROI improvement target.

The next investment should target post-resolution outcomes — dispute prediction, relief 
classification, and real-time bureau monitoring — not response speed, which is essentially a 
solved problem.